# Inferencia con `modelo_v2_mejor.keras`

Notebook para cargar el modelo entrenado y hacer predicciones sobre imágenes nuevas (NORMAL vs PNEUMONIA). Ajusta la ruta del modelo y las rutas de imagen/directorio antes de ejecutar.


## 1. Configuración
- Ajusta `MODEL_PATH` a la ubicación real del modelo guardado (`modelo_v2_mejor.keras`).
- Define `THRESHOLD` si quieres más sensibilidad (menor umbral) o más precisión (mayor umbral).


In [1]:
import os, numpy as np, tensorflow as tf
from PIL import Image
import matplotlib.pyplot as plt

# Ruta al modelo entrenado (ajusta según dónde lo guardaste)
MODEL_PATH = r".\modelo_v2_mejor.keras"

# Configuración de entrada
IMG_SIZE = (180, 180)
THRESHOLD = 0.5
CLASS_NAMES = ["NORMAL", "PNEUMONIA"]

assert os.path.isfile(MODEL_PATH), f"No se encontró el modelo en: {MODEL_PATH}"
model = tf.keras.models.load_model(MODEL_PATH)
print("Modelo cargado:", MODEL_PATH)


Modelo cargado: .\modelo_v2_mejor.keras


## 2. Funciones de utilidad
- `load_image`: carga y normaliza la imagen.
- `predict_image`: devuelve probabilidad y clase.
- Usa `THRESHOLD` para decidir la clase (por defecto 0.5).


In [2]:
def load_image(path):
    img = Image.open(path).convert("RGB").resize(IMG_SIZE)
    arr = np.array(img) / 255.0
    return np.expand_dims(arr, axis=0), img

def predict_image(path, threshold=THRESHOLD):
    x, img = load_image(path)
    prob = float(model.predict(x, verbose=0)[0][0])
    label = CLASS_NAMES[1] if prob >= threshold else CLASS_NAMES[0]
    return prob, label, img


## 3. Inferencia sobre una imagen (con carga en notebook)
Usa el botón para subir una imagen (jpg/png). La celda mostrará la probabilidad y la imagen. Si prefieres usar una ruta fija, ajusta `image_path` abajo del todo.


In [3]:
import io, ipywidgets as widgets
from IPython.display import display

upload_single = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
out_single = widgets.Output()


def on_upload_single(change):
    if not upload_single.value:
        return
    name, file_info = next(iter(upload_single.value.items()))
    img = Image.open(io.BytesIO(file_info['content'])).convert('RGB')
    img_resized = img.resize(IMG_SIZE)
    arr = np.array(img_resized) / 255.0
    x = np.expand_dims(arr, axis=0)
    prob = float(model.predict(x, verbose=0)[0][0])
    label = CLASS_NAMES[1] if prob >= THRESHOLD else CLASS_NAMES[0]
    with out_single:
        out_single.clear_output()
        print(f"Archivo: {name}")
        print(f"Prob. PNEUMONIA: {prob:.4f} | Clase: {label} | Umbral: {THRESHOLD}")
        display(img)
    upload_single.value.clear()

upload_single.observe(on_upload_single, names='value')

display(widgets.VBox([
    widgets.HTML('<b>Sube una imagen (.jpg/.png) para predecir:</b>'),
    upload_single,
    out_single
]))


## 3.b Interfaz con botón de carga (notebook)
Usa un botón para subir una imagen (jpg/png) y ajustar el umbral sobre la marcha. Al mover el slider se recalcula la predicción de la última imagen cargada.


In [4]:
import io
import ipywidgets as widgets
from IPython.display import display

uploader = widgets.FileUpload(accept='.jpg,.jpeg,.png', multiple=False)
threshold_slider = widgets.FloatSlider(min=0.3, max=0.7, step=0.01, value=THRESHOLD, description='Umbral')
btn_clear = widgets.Button(description='Limpiar', button_style='info')
out = widgets.Output()
state = {'last_bytes': None, 'last_name': None}


def run_inference(file_bytes, filename):
    img = Image.open(io.BytesIO(file_bytes)).convert('RGB')
    img_resized = img.resize(IMG_SIZE)
    arr = np.array(img_resized) / 255.0
    x = np.expand_dims(arr, axis=0)
    prob = float(model.predict(x, verbose=0)[0][0])
    label = CLASS_NAMES[1] if prob >= threshold_slider.value else CLASS_NAMES[0]
    with out:
        out.clear_output()
        print(f"Archivo: {filename}")
        print(f"Prob. PNEUMONIA: {prob:.4f} | Clase: {label} | Umbral: {threshold_slider.value}")
        display(img)
    state['last_bytes'] = file_bytes
    state['last_name'] = filename


def on_upload_change(change):
    if not uploader.value:
        return
    name, file_info = next(iter(uploader.value.items()))
    run_inference(file_info['content'], name)
    uploader.value.clear()


def on_threshold_change(change):
    if state['last_bytes'] is not None:
        run_inference(state['last_bytes'], state['last_name'])


def on_clear(_):
    out.clear_output()
    state['last_bytes'] = None
    state['last_name'] = None


uploader.observe(on_upload_change, names='value')
threshold_slider.observe(on_threshold_change, names='value')
btn_clear.on_click(on_clear)

display(widgets.VBox([
    widgets.HTML('<b>Carga una imagen y ajusta el umbral si quieres más sensibilidad.</b>'),
    widgets.HBox([uploader, threshold_slider, btn_clear]),
    out
]))


## 4. Inferencia por lote en un directorio (opcional)
- Procesa todas las imágenes .jpg/.jpeg/.png de un folder.
- Ajusta `folder_path` y revisa el listado de resultados.


In [5]:
import glob

def predict_dir(folder_path, threshold=THRESHOLD):
    patterns = ['*.jpg', '*.jpeg', '*.png']
    files = []
    for p in patterns:
        files.extend(glob.glob(os.path.join(folder_path, p)))
    results = []
    for f in files:
        prob, label, _ = predict_image(f, threshold=threshold)
        results.append((f, prob, label))
    return results

# Cambia esta ruta a tu carpeta
folder_path = r"RUTA\A\CARPETA_CON_IMAGENES"

if os.path.isdir(folder_path):
    preds = predict_dir(folder_path)
    for f, prob, label in preds[:20]:  # muestra los primeros 20
        print(f"{label:10s} prob={prob:.4f} -> {f}")
    print(f"Total procesadas: {len(preds)}")
else:
    print("Actualiza 'folder_path' con una ruta válida.")


Actualiza 'folder_path' con una ruta válida.


## 5. Ajuste de umbral (opcional)
- Si quieres más sensibilidad, prueba `THRESHOLD = 0.45` o `0.40` y vuelve a correr las celdas de inferencia.
- Si quieres más precisión, súbelo (ej. 0.55).
